# 03 — Perbandingan Klasifikasi

Lima algoritma klasifikasi dibandingkan pada data, fitur, praproses, split,
lipatan CV, dan metrik yang sama:

**Logistic Regression, Random Forest, Gradient Boosting, XGBoost, KNN.**

Target adalah label biner `Mahal`/`Murah` yang dibentuk dari median harga per
kota. Tidak ada hyperparameter tuning; semua model memakai nilai bawaan
(kecuali `random_state`, `n_jobs=1`, dan `max_iter` Logistic Regression agar
konvergen).

In [1]:
import importlib.util
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

if importlib.util.find_spec("xgboost") is None:
    print("Memasang xgboost (diperlukan notebook model)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

cwd = Path.cwd()
module_path = cwd / "perbandingan_common.py"
if not module_path.exists():
    docs_module = cwd / "docs" / "perbandingan_common.py"
    if docs_module.exists():
        sys.path.insert(0, str(docs_module.parent))
    else:
        url = "https://raw.githubusercontent.com/RobbyCatur/rumahpoint/main/docs/perbandingan_common.py"
        print("Mengunduh modul bersama dari GitHub ...")
        urllib.request.urlretrieve(url, module_path)
        sys.path.insert(0, str(cwd))
else:
    sys.path.insert(0, str(cwd))

import perbandingan_common as c

print("modul:", c.__file__)
print("repo terdeteksi:", c.REPO_ROOT)
print("direktori keluaran:", c.OUTPUT_DIR)
display(pd.DataFrame([c.describe_environment()]).T.rename(columns={0: "nilai"}))

modul: /opt/rumahpoint/docs/perbandingan_common.py
repo terdeteksi: /opt/rumahpoint
direktori keluaran: /opt/rumahpoint/docs/perbandingan_output


,nilai
python,3.12.3
platform,Linux-7.0.14-28-pve-x86_64-with-glibc2.39
sklearn,1.9.0
pandas,3.0.5
numpy,2.5.2
joblib,1.6.0
xgboost,3.4.1


## 1. Hubungan dengan tiga rujukan

Ketiga rujukan di notebook regresi meneliti **regresi harga**, bukan klasifikasi
Murah/Mahal. Karena itu tidak ada angka akurasi jurnal yang bisa dibandingkan
langsung di sini. Klasifikasi adalah perluasan Rumah Point: aplikasi juga
menampilkan kategori harga.

Yang bisa diambil dari jurnal: pemilihan model tidak stabil antar-dataset, dan
Random Forest sering menjadi pesaing kuat. Bagian ini menguji apakah pola itu
bertahan pada dataset multi-kota yang lebih besar.

Label dibentuk dari median per kota pada seluruh data bersih, mengikuti
`database/seed.py`. Karena ambangnya median, distribusi kelas mendekati
seimbang; baseline kelas mayoritas tetap ditampilkan sebagai pembanding.

Ambang median memakai seluruh data bersih, termasuk baris uji, mengikuti
`database/seed.py`; label tidak sepenuhnya out-of-sample.

## 2. Protokol yang sama untuk semua model

| Komponen | Keputusan |
| --- | --- |
| Data | CSV union setelah pembersihan notebook 01; baris latih/uji tetap |
| Split hold-out | 80% latih / 20% uji, `random_state=42` |
| CV | 5 lipatan `KFold(shuffle=True, random_state=42)` pada baris latih |
| Fitur | `luas_m2`, `luas_bangunan`, `jumlah_kamar`, `jumlah_kamar_mandi`, `lokasi` |
| Target | `label_klasifikasi` (1 = Mahal, 0 = Murah) |
| Praproses | imputasi median + `StandardScaler` untuk numerik; OneHot `lokasi`; fit di dalam pipeline per lipatan |
| Model | lima model, parameter bawaan, tanpa tuning |
| Metrik hold-out | akurasi, precision/recall/F1 macro, F1 weighted, ROC-AUC |
| Metrik CV | rata-rata dan simpangan baku metrik yang sama |
| Waktu | waktu latih dan prediksi hold-out; rata-rata CV |
| Ukuran model | hasil `joblib.dump` tanpa kompresi |
| Aturan pemenang | rata-rata F1-macro CV tertinggi; tie-break ROC-AUC CV; lalu F1-macro hold-out |

KNN memakai seluruh baris latih agar data identik dengan model lain.

In [2]:
df, rep, split, train_idx, test_idx, folds = c.load_bundle()
train_df, test_df = c.train_test_frames(df, train_idx, test_idx)
X_train = c.feature_frame(train_df)
X_test = c.feature_frame(test_df)
y_train = train_df[c.TARGET_CLF]
y_test = test_df[c.TARGET_CLF]
print(f"latih {len(X_train):,}; uji {len(X_test):,}")
display(y_train.value_counts(normalize=True).rename_axis("kelas").to_frame("proporsi"))
baseline = float(y_train.value_counts(normalize=True).max())
print(f"baseline akurasi kelas mayoritas: {baseline:.4f}")
display(X_train.head(3))

=== SIDIK JARI DATA ===
berkas dataset        : /opt/rumahpoint/dataset/properti_multi_kota.csv
baris mentah          : 64,650
baris bersih          : 64,228
baris latih / uji     : 51,382 / 12,846
hash baris latih      : cfe3defcae087f44fa6a7189a319ef258f4138318a7532b315bf80a85dc9990e
hash baris uji        : 718f7f914fe83904e4899da30d2fc6b7a06e249431befe26985f93e43efe7aef
hash file dataset     : 34375406a6d72b21bad55e5784caddc931a85bb20c999414d3fdd6b2f5774d18
lipatan 0 (indeks uji): f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5d6798761cf3ae938d4
lipatan 1 (indeks uji): c177e4c57ee746c9d021815cecfbc4e8128469d08d1617f086033940cee7efe6
lipatan 2 (indeks uji): 7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21beddf20741479178c025e
lipatan 3 (indeks uji): 9956fdc032bc252aeb7d0ecac5531626bac860dee609478a49110408ee8b19cd
lipatan 4 (indeks uji): a0b04506e601150298462828e0306e977fd83ad22063aa0ebd10d9663b7c51dc


latih 51,382; uji 12,846


,proporsi
kelas,
1,0.508466
0,0.491534


baseline akurasi kelas mayoritas: 0.5085


,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,lokasi
0,83.0,60.0,2.0,2.0,Kabupaten Karimun
1,108.0,70.0,2.0,2.0,Kabupaten Sigi
2,66.0,40.0,2.0,1.0,Kabupaten Gresik


### Parameter tiap model

In [3]:
models = c.classification_models()
display(c.model_parameter_table(models))

,model,C,solver,max_iter,random_state,n_jobs,n_estimators,max_depth,learning_rate,eval_metric,n_neighbors,weights
0,Logistic Regression,1.0,lbfgs,2000.0,42.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Random Forest,NaN,NaN,NaN,42.0,1.0,100.0,NaN,NaN,NaN,NaN,NaN
2,Gradient Boosting,NaN,NaN,NaN,42.0,NaN,100.0,3.0,0.1,NaN,NaN,NaN
3,XGBoost,NaN,NaN,NaN,42.0,1.0,NaN,NaN,NaN,logloss,NaN,NaN
4,KNN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,5.0,uniform


## 3. Hasil hold-out

In [4]:
rows = []
preds = {}
for name, factory in models.items():
    row, pred, _ = c.evaluate_holdout(
        name, factory, X_train, y_train, X_test, y_test, task="klasifikasi"
    )
    rows.append(row)
    preds[name] = pred
holdout = pd.DataFrame(rows)
c.save_csv("classification_results_holdout.csv", holdout)
holdout_view = holdout[[
    "model", "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "f1_weighted", "roc_auc", "fit_seconds", "predict_seconds", "model_size_mb",
]]
display(holdout_view.style.format({
    "accuracy": "{:.4f}",
    "precision_macro": "{:.4f}",
    "recall_macro": "{:.4f}",
    "f1_macro": "{:.4f}",
    "f1_weighted": "{:.4f}",
    "roc_auc": "{:.4f}",
    "fit_seconds": "{:.2f}",
    "predict_seconds": "{:.3f}",
    "model_size_mb": "{:.2f}",
}))
print(f"pembanding: baseline akurasi kelas mayoritas = {baseline:.4f}")

[hold-out] Logistic Regression: akurasi=0.7769, F1_macro=0.7757, ROC-AUC=0.8666, latih=10.9s, prediksi=3.0s, ukuran=0.01 MB


[hold-out] Random Forest: akurasi=0.8641, F1_macro=0.8640, ROC-AUC=0.9376, latih=54.6s, prediksi=6.5s, ukuran=144.64 MB


[hold-out] Gradient Boosting: akurasi=0.7949, F1_macro=0.7949, ROC-AUC=0.8734, latih=192.6s, prediksi=0.7s, ukuran=0.15 MB


[hold-out] XGBoost: akurasi=0.8325, F1_macro=0.8325, ROC-AUC=0.9157, latih=11.9s, prediksi=1.6s, ukuran=0.28 MB


[hold-out] KNN: akurasi=0.8346, F1_macro=0.8346, ROC-AUC=0.9046, latih=8.2s, prediksi=38.4s, ukuran=172.50 MB


[tulis] /opt/rumahpoint/docs/perbandingan_output/classification_results_holdout.csv


,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,roc_auc,fit_seconds,predict_seconds,model_size_mb
0,Logistic Regression,0.7769,0.7868,0.7789,0.7757,0.7754,0.8666,10.94,2.975,0.01
1,Random Forest,0.8641,0.8641,0.8640,0.8640,0.8641,0.9376,54.62,6.492,144.64
2,Gradient Boosting,0.7949,0.7955,0.7954,0.7949,0.7949,0.8734,192.56,0.694,0.15
3,XGBoost,0.8325,0.8326,0.8328,0.8325,0.8325,0.9157,11.93,1.602,0.28
4,KNN,0.8346,0.8346,0.8347,0.8346,0.8346,0.9046,8.23,38.404,172.50


pembanding: baseline akurasi kelas mayoritas = 0.5085


## 4. Hasil validasi silang 5 lipatan

In [5]:
fold_frames = []
summaries = []
for name, factory in models.items():
    folds_df, summary = c.evaluate_cv(
        name, factory, X_train, y_train, folds, task="klasifikasi"
    )
    folds_df.insert(0, "model", name)
    fold_frames.append(folds_df)
    summaries.append(summary)
cv_folds = pd.concat(fold_frames, ignore_index=True)
cv = pd.DataFrame(summaries)
c.save_csv("classification_results_cv_folds.csv", cv_folds)
c.save_csv("classification_results_cv.csv", cv)
cv_view = cv[[
    "model", "accuracy_mean", "accuracy_std", "f1_macro_mean", "f1_macro_std",
    "precision_macro_mean", "recall_macro_mean", "roc_auc_mean",
    "fit_seconds_mean", "predict_seconds_mean",
]]
display(cv_view.style.format({
    "accuracy_mean": "{:.4f}",
    "accuracy_std": "{:.4f}",
    "f1_macro_mean": "{:.4f}",
    "f1_macro_std": "{:.4f}",
    "precision_macro_mean": "{:.4f}",
    "recall_macro_mean": "{:.4f}",
    "roc_auc_mean": "{:.4f}",
    "fit_seconds_mean": "{:.2f}",
    "predict_seconds_mean": "{:.3f}",
}))

[cv Logistic Regression] lipatan 0: akurasi=0.7884, F1_macro=0.7876


[cv Logistic Regression] lipatan 1: akurasi=0.7818, F1_macro=0.7808


[cv Logistic Regression] lipatan 2: akurasi=0.7860, F1_macro=0.7850


[cv Logistic Regression] lipatan 3: akurasi=0.7855, F1_macro=0.7847


[cv Logistic Regression] lipatan 4: akurasi=0.7836, F1_macro=0.7826


[cv] Logistic Regression: akurasi=0.7851 ± 0.0022, F1_macro=0.7841 ± 0.0023


... (23 outputs left out)

[cv] KNN: akurasi=0.8316 ± 0.0025, F1_macro=0.8316 ± 0.0025


[tulis] /opt/rumahpoint/docs/perbandingan_output/classification_results_cv_folds.csv


[tulis] /opt/rumahpoint/docs/perbandingan_output/classification_results_cv.csv


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,precision_macro_mean,recall_macro_mean,roc_auc_mean,fit_seconds_mean,predict_seconds_mean
0,Logistic Regression,0.7851,0.0022,0.7841,0.0023,0.7927,0.7864,0.8700,7.99,0.239
1,Random Forest,0.8648,0.0026,0.8647,0.0026,0.8649,0.8647,0.9382,34.56,2.519
2,Gradient Boosting,0.8026,0.0059,0.8026,0.0059,0.8029,0.8029,0.8830,93.76,0.595
3,XGBoost,0.8376,0.0037,0.8376,0.0037,0.8376,0.8377,0.9208,8.02,0.247
4,KNN,0.8316,0.0025,0.8316,0.0025,0.8316,0.8316,0.9028,4.23,21.276


## 5. Pemenang dan alasan

Aturan sudah ditetapkan sebelum melihat hasil: rata-rata F1-macro CV tertinggi,
tie-break ROC-AUC CV, lalu F1-macro hold-out.

In [6]:
combined = holdout.merge(cv, on="model", suffixes=("", "_cv"))
winner = c.choose_classification_winner(combined)
print("Pemenang:", winner["model"])
print("Alasan:", winner["reason"])
peringkat = pd.DataFrame({
    "peringkat": range(1, len(winner["ranking"]) + 1),
    "model": winner["ranking"],
})
display(peringkat)

Pemenang: Random Forest
Alasan: Rata-rata F1-macro CV tertinggi (0.8647), ROC-AUC CV 0.9382; hold-out akurasi 0.8641 dan F1-macro 0.8640.


,peringkat,model
0,1,Random Forest
1,2,XGBoost
2,3,KNN
3,4,Gradient Boosting
4,5,Logistic Regression


## 6. Matriks kebingungan model terbaik

Matriks dihitung pada prediksi hold-out model pemenang, memakai baris di
`preds` yang baru saja dihasilkan.

In [7]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds[winner["model"]], labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Aktual Murah", "Aktual Mahal"],
    columns=["Prediksi Murah", "Prediksi Mahal"],
)
display(cm_df)

,Prediksi Murah,Prediksi Mahal
Aktual Murah,5392,890
Aktual Mahal,856,5708


## 7. Simpan ringkasan

In [8]:
summary = {
    "task": "klasifikasi",
    "winner": winner,
    "protocol": {
        "n_rows": rep["clean_rows"],
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "split_seed": split["seed"],
        "test_size": split["test_size"],
        "cv_folds": len(folds),
        "features": c.MAIN_FEATURES,
        "target": c.TARGET_CLF,
        "preprocessing": "median imputer + StandardScaler (numerik), OneHot (lokasi), fit per lipatan",
        "models": list(models),
        "winner_rule": "mean CV F1-macro tertinggi; tie-break mean ROC-AUC CV; lalu F1-macro hold-out",
    },
    "sidik_jari": {
        "raw_rows": rep["raw_rows"],
        "clean_rows": rep["clean_rows"],
        "train_sha256": split["train_sha256"],
        "test_sha256": split["test_sha256"],
    },
    "baseline_akurasi_kelas_mayoritas": baseline,
    "holdout": holdout.to_dict("records"),
    "cv": cv.to_dict("records"),
}
c.save_json("classification_summary.json", summary)
print("Berkas hasil klasifikasi selesai ditulis.")

[tulis] /opt/rumahpoint/docs/perbandingan_output/classification_summary.json


Berkas hasil klasifikasi selesai ditulis.


## Catatan membaca hasil

- Akurasi harus dibandingkan dengan baseline kelas mayoritas, bukan dengan 0,5.
- F1-macro memberi bobot sama pada kedua kelas; ROC-AUC mengukur pemeringkatan
  probabilitas dan tidak bergantung pada ambang 0,5.
- Label diturunkan dari harga, jadi hasil klasifikasi tidak independen dari
  kualitas regresi.
- Model produksi tidak diubah oleh notebook ini.